# 损失函数

In [1]:
import tvm.testing
from tvm import relax
from tvm.ir.base import assert_structural_equal
from tvm.script import relax as R, ir as I

@I.ir_module
class Module:
    @R.function
    def forward(
        x: R.Tensor((2, 4), "float32"),
        w: R.Tensor((4, 4), "float32"),
        b: R.Tensor((2, 4), "float32"),
    ) -> R.Tensor((2, 4), "float32"):
        with R.dataflow():
            lv: R.Tensor((2, 4), "float32") = R.matmul(x, w)
            out: R.Tensor((2, 4), "float32") = R.add(lv, b)
            R.output(out)
        return out

## L1 损失

In [2]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N, C), "float32")
l1_loss = relax.training.loss.L1Loss()
l1_loss(predictions, targets).show()

In [3]:
s = Module["forward"].ret_struct_info
l1_loss = relax.training.loss.L1Loss(reduction="sum")
After = relax.training.AppendLoss("forward", l1_loss(s, s), l1_loss.num_backbone_outputs)(
    Module
)
After["forward_loss"].show()

## MSE 损失

In [4]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N, C), "float32")
mse_loss = relax.training.loss.MSELoss()
mse_loss(predictions, targets).show()

In [5]:
s = Module["forward"].ret_struct_info
mse_loss = relax.training.loss.MSELoss(reduction="sum")
After = relax.training.AppendLoss("forward", mse_loss(s, s), mse_loss.num_backbone_outputs)(
    Module
)
After["forward_loss"].show()

## 交叉熵损失

带有权重：

In [6]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N,), "int64")
weights = relax.TensorStructInfo((C,), "float32")
cross_entropy_loss = relax.training.loss.CrossEntropyLoss(reduction="sum", ignore_index=1)
cross_entropy_loss(predictions, targets, weights).show()

无权重：

In [7]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N,), "int64")
cross_entropy_loss = relax.training.loss.CrossEntropyLoss()
cross_entropy_loss(predictions, targets).show()

In [8]:
s = Module["forward"].ret_struct_info
N = s.shape[0]
C = s.shape[1]
targets = relax.TensorStructInfo((N,), "int64")
weights = relax.TensorStructInfo((C,), "float32")
cross_entropy_loss = relax.training.loss.CrossEntropyLoss(reduction="sum", ignore_index=1)
After = relax.training.AppendLoss(
    "forward", cross_entropy_loss(s, targets, weights), cross_entropy_loss.num_backbone_outputs
)(Module)
After["forward_loss"].show()

## 分类交叉熵损失

In [9]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N, C), "int64")
weights = relax.TensorStructInfo((C,), "float32")
categorical_cross_entropy_loss = relax.training.loss.CategoricalCrossEntropyLoss(
    reduction="sum"
)
categorical_cross_entropy_loss(predictions, targets, weights).show()

In [10]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N, C), "int64")
categorical_cross_entropy_loss = relax.training.loss.CategoricalCrossEntropyLoss()
categorical_cross_entropy_loss(predictions, targets).show()

忽略目标值为 1 的类别：

In [11]:
N = 3
C = 5
predictions = relax.TensorStructInfo((N, C), "float32")
targets = relax.TensorStructInfo((N, C), "int64")
weights = relax.TensorStructInfo((C,), "float32")
categorical_cross_entropy_loss = relax.training.loss.CategoricalCrossEntropyLoss(
    reduction="sum", ignore_index=1
)
categorical_cross_entropy_loss(predictions, targets, weights).show()